# 03 - Submission
Memuat pipeline final, memprediksi `test.csv`, dan menulis `submissions/submission.csv` (kolom sama dengan `sample_submission.csv`, `index=False`).

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
sys.path.insert(0, str(Path.cwd().parent))  # agar `scripts` bisa di-import dari folder notebooks/
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scripts.config import *
from scripts.preprocess import load_data, encode_target
sns.set_theme(style='whitegrid')
import joblib
from scripts.preprocess import decode_target
_, test = load_data()
pipe = joblib.load(MODEL_DIR/'pipeline_model_lgbm.pkl')
pred = decode_target(pipe.predict(test))
sample = pd.read_csv(DATA_DIR/'sample_submission.csv')
submission = sample[[ID_COL]].merge(pd.DataFrame({ID_COL: test[ID_COL], TARGET: pred}), on=ID_COL, how='left')
submission[TARGET] = submission[TARGET].str.replace('Grade ', '', regex=False).astype(int)  # evaluator Kaggle meminta angka 1-5
assert len(submission) == len(sample) and submission[TARGET].notna().all() and list(submission.columns) == list(sample.columns)
submission.to_csv(SUBMISSION_DIR/'submission.csv', index=False, lineterminator='\r\n')
submission.head()

,building_id,damage_grade
0,122207000141,2
1,284909000871,5
2,241904000231,4
3,248308000891,2
4,300405002981,4


In [2]:
print(submission[TARGET].value_counts(normalize=True).sort_index().round(3))
print('Distribusi train:')
print(pd.read_csv(DATA_DIR/'train.csv')[TARGET].str[-1].astype(int).value_counts(normalize=True).sort_index().round(3))

damage_grade
1    0.117
2    0.101
3    0.174
4    0.237
5    0.370
Name: proportion, dtype: float64
Distribusi train:


damage_grade
1    0.103
2    0.114
3    0.179
4    0.241
5    0.362
Name: proportion, dtype: float64


Distribusi prediksi mendekati distribusi train, jadi tidak ada pergeseran kelas yang mencurigakan. Upload `submissions/submission.csv` ke Kaggle.